# 07 · Complete & complex queries

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~40 min**

## Learning objectives
- compute **proportions** with a window aggregate (`SUM(...) OVER (...)`);
- find the **top item per group** with `RANK() OVER (PARTITION BY ...)`;
- chain steps with **multiple CTEs**;
- compare groups with **conditional aggregation** (`SUM(CASE WHEN ...)`).

Now we combine what we have seen: `JOIN`, `GROUP BY`, subqueries and CTEs, and window functions, into the kind of complete analytical queries you would write to analyse a study. Read each one clause by clause. They are longer, but every piece is something you already met.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [ ]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

## 1. Relative abundance *within* each sample
Counts depend on sequencing depth, so we usually want percentages per sample. A window aggregate divides each phylum's reads by that sample's total, with no separate query needed.

In [ ]:
%%sql
SELECT a.sample_id, t.phylum,
       SUM(a.count) AS reads,
       ROUND(100.0 * SUM(a.count)
             / SUM(SUM(a.count)) OVER (PARTITION BY a.sample_id), 1) AS pct_of_sample
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY a.sample_id, t.phylum
ORDER BY a.sample_id, pct_of_sample DESC;

## 2. The dominant genus in each sample
`RANK()` numbers rows within each sample; we keep rank 1 in an outer query (a CTE makes it readable).

In [ ]:
%%sql
WITH ranked AS (
    SELECT a.sample_id, t.genus, a.count,
           RANK() OVER (PARTITION BY a.sample_id ORDER BY a.count DESC) AS rnk
    FROM abundance a
    JOIN taxa t ON a.taxon_id = t.taxon_id
)
SELECT sample_id, genus AS dominant_genus, count
FROM ranked
WHERE rnk = 1
ORDER BY sample_id;

A different window twist: `DENSE_RANK` at the environment level finds each environment's most-read phylum without gaps in the ranking.

In [ ]:
%%sql
-- most abundant phylum in each environment (dense_rank keeps 1,2,2,3...)
WITH ep AS (
    SELECT s.environment, t.phylum, SUM(a.count) AS reads,
           DENSE_RANK() OVER (PARTITION BY s.environment ORDER BY SUM(a.count) DESC) AS rnk
    FROM abundance a
    JOIN samples s ON a.sample_id = s.sample_id
    JOIN taxa t    ON a.taxon_id  = t.taxon_id
    GROUP BY s.environment, t.phylum
)
SELECT environment, phylum, reads FROM ep
WHERE rnk = 1
ORDER BY environment;

## 3. Phylum composition (%) per environment — chained CTEs
First total reads per environment × phylum, then the environment totals, then divide. Three readable steps.

In [ ]:
%%sql
WITH env_phylum AS (
    SELECT s.environment, t.phylum, SUM(a.count) AS reads
    FROM abundance a
    JOIN samples s ON a.sample_id = s.sample_id
    JOIN taxa t    ON a.taxon_id  = t.taxon_id
    GROUP BY s.environment, t.phylum
),
env_total AS (
    SELECT environment, SUM(reads) AS total FROM env_phylum GROUP BY environment
)
SELECT ep.environment, ep.phylum,
       ROUND(100.0 * ep.reads / et.total, 1) AS pct
FROM env_phylum ep
JOIN env_total et ON ep.environment = et.environment
ORDER BY ep.environment, pct DESC;

## 4. Treatment effect — fold-change per phylum (conditional aggregation)
`SUM(CASE WHEN ...)` totals only the rows that match a condition, so we get Control and Amended side by side in one row and can divide them.

In [ ]:
%%sql
SELECT t.phylum,
       SUM(CASE WHEN s.treatment = 'Control' THEN a.count ELSE 0 END) AS control_reads,
       SUM(CASE WHEN s.treatment = 'Amended' THEN a.count ELSE 0 END) AS amended_reads,
       ROUND(1.0 * SUM(CASE WHEN s.treatment = 'Amended' THEN a.count ELSE 0 END)
             / NULLIF(SUM(CASE WHEN s.treatment = 'Control' THEN a.count ELSE 0 END), 0), 2)
             AS amended_vs_control
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
GROUP BY t.phylum
ORDER BY amended_vs_control DESC;

Conditional aggregation also compares environments side by side, not just treatments.

In [ ]:
%%sql
-- Soil vs. Freshwater reads per phylum in one row each (conditional SUM)
SELECT t.phylum,
       SUM(CASE WHEN s.environment = 'Soil' THEN a.count ELSE 0 END) AS soil_reads,
       SUM(CASE WHEN s.environment = 'Freshwater' THEN a.count ELSE 0 END) AS freshwater_reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
GROUP BY t.phylum
ORDER BY soil_reads DESC;

## 5. Prevalence and mean abundance together
Several aggregates in one query, filtered with `HAVING`, give a compact profile of the common genera.

In [ ]:
%%sql
SELECT t.genus,
       COUNT(DISTINCT a.sample_id) AS n_samples,
       ROUND(AVG(a.count), 1) AS mean_when_present,
       SUM(a.count) AS total_reads
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY t.genus
HAVING n_samples >= 20
ORDER BY total_reads DESC;

---
## Exercises

**Exercise.** Show the relative abundance (%) of each phylum **within Sediment samples only** (a window over the whole result with `OVER ()`).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT t.phylum, SUM(a.count) AS reads,
       ROUND(100.0 * SUM(a.count) / SUM(SUM(a.count)) OVER (), 1) AS pct
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
WHERE s.environment = 'Sediment'
GROUP BY t.phylum
ORDER BY pct DESC;
```
</details>

**Exercise.** List the **top 3 genera by reads in each environment** (RANK partitioned by environment).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
WITH r AS (
  SELECT s.environment, t.genus, SUM(a.count) AS reads,
         RANK() OVER (PARTITION BY s.environment ORDER BY SUM(a.count) DESC) AS rnk
  FROM abundance a
  JOIN samples s ON a.sample_id = s.sample_id
  JOIN taxa t    ON a.taxon_id  = t.taxon_id
  GROUP BY s.environment, t.genus)
SELECT environment, genus, reads FROM r WHERE rnk <= 3
ORDER BY environment, rnk;
```
</details>

**Exercise.** For each environment give the number of samples, the mean pH and the mean library size (a CTE for library size, then join + several aggregates).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
WITH lib AS (SELECT sample_id, SUM(count) AS reads FROM abundance GROUP BY sample_id)
SELECT s.environment, COUNT(*) AS n_samples,
       ROUND(AVG(s.ph), 2) AS mean_ph,
       ROUND(AVG(l.reads)) AS mean_library_size
FROM samples s
JOIN lib l ON s.sample_id = l.sample_id
GROUP BY s.environment;
```
</details>

### Recap
- A window aggregate `SUM(x) OVER (PARTITION BY …)` gives proportions without a second query.
- `RANK() OVER (PARTITION BY … ORDER BY …)` plus an outer filter gives top-N per group.
- Multiple CTEs (`WITH a AS (…), b AS (…)`) turn a hard query into readable steps.
- `SUM(CASE WHEN … )` compares groups side by side (conditional aggregation).
- Next: 08 · Hands-on capstone.